# Lecture 7. DocumentGPT

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

llm = ChatOpenAI(
    temperature=0.1,
)

ind_check_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            질문 텍스트에 개인정보가 있는지 확인해야합니다. 
            
            개인정보 형태는 아래와 같습니다.
            1. 주민등록번호 : 숫자 13자리로 구성되며, '-'문자가 중간에 있을 수 있습니다. 
            2. 이메일 : 영문 텍스트로 되어 있으며 '@'와 도메인이 포함되어 있습니다.
            
            개인정보가 있으면 
            답변 : ''
            -------
            """,
        ),
        ("human", "{question}"),
    ]
)

ind_check_chain = ind_check_prompt | llm
ind_check_chain.invoke({"question": RunnablePassthrough()})

final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            질문 내용에 개인정보가 있는지 확인하고, 개인정보가 있다면 어떤 것이 위반인지 답변을 주어야 합니다.
            모르는 질문에는 답변하지 마세요.
            
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

chain = {"context": ind_check_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("안녕하세요. 저는 1234561234567입니다.")

AIMessage(content='개인정보가 포함되어 있습니다. 이는 주민등록번호로 보이며, 이는 개인정보 보호법에 의해 보호되어야 합니다. 개인정보를 포함한 내용은 공개되지 않도록 주의해주시기 바랍니다.', response_metadata={'token_usage': <OpenAIObject at 0x126267a10> JSON: {
  "prompt_tokens": 226,
  "completion_tokens": 79,
  "total_tokens": 305
}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-71777250-872c-4a2e-a9f5-aeedc80f0f48-0')